# Conformal Correction Across Forecaster Classes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/Conformal_Oracle/blob/main/CO_conformal_garch_ewma/CO_conformal_garch_ewma.ipynb)

This notebook applies the **identical conformal calibration procedure** used for LLM forecasters
to GARCH and EWMA benchmarks, demonstrating the framework's **forecaster-agnosticism**.

**Paper:** *Distribution-Free Recalibration of Tail Quantile Forecasts under Temporal Dependence*

**Repository:** [QuantLet/Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle)


## 1. Setup


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Parameters (identical to LLM pipeline)
ALPHA_VAR = 0.01
ALPHA_ES  = 0.025
F_CAL     = 0.70

print('Setup complete. Parameters:')
print(f'  alpha_VaR = {ALPHA_VAR}, alpha_ES = {ALPHA_ES}, f_cal = {F_CAL}')


Setup complete. Parameters:
  alpha_VaR = 0.01, alpha_ES = 0.025, f_cal = 0.7


## 2. Conformal Calibration Procedure

The conformal correction is applied identically to all forecasters:

1. **Split**: first 70% = calibration, last 30% = test
2. **VaR nonconformity scores**: $s_t^V = \hat{q}_t^{lo} - r_t$ where $\hat{q}_t^{lo} = -\text{VaR}_t$
3. **Conformal threshold**: $\hat{q}_V = \text{Quantile}(\{s_t^V\}_{\text{cal}};\, 1-\alpha)$
4. **Corrected VaR**: $\text{VaR}_t^{cp} = -(\hat{q}_t^{lo} - \hat{q}_V) = \text{VaR}_t^{raw} + \hat{q}_V$
5. **ES correction**: on violation days, $s_t^E = r_t + \text{ES}_t$, then $\hat{q}_E$ analogously

The key insight: $\hat{q}_V$ measures **how much the forecaster underestimates tail risk**.


## 3. Core Conformal Function


In [2]:
def apply_conformal(q_lo, raw_var, raw_es, realized,
                    alpha_var=0.01, alpha_es=0.025, f_cal=0.70):
    """Apply conformal correction to pre-computed VaR/ES forecasts."""
    T = len(q_lo)
    n_cal = int(T * f_cal)

    # VaR nonconformity scores: s_t = q_lo_t - r_t
    s_V = np.array([q_lo[t] - realized[t] if not (np.isnan(q_lo[t]) or np.isnan(realized[t]))
                    else np.nan for t in range(n_cal)])
    s_V_valid = s_V[~np.isnan(s_V)]

    if len(s_V_valid) > 0:
        q_level = min(np.ceil((len(s_V_valid)+1)*(1-alpha_var)) / len(s_V_valid), 1.0)
        q_hat_V = np.quantile(s_V_valid, q_level)
    else:
        q_hat_V = 0.0

    # ES scores on violation days
    s_E = [realized[t] + raw_es[t] for t in range(n_cal)
           if not (np.isnan(realized[t]) or np.isnan(raw_var[t]))
           and realized[t] < -raw_var[t]]
    s_E = np.array(s_E)
    if len(s_E) > 0:
        q_level_E = min(np.ceil((len(s_E)+1)*(1-alpha_es)) / len(s_E), 1.0)
        q_hat_E = np.quantile(s_E, q_level_E)
    else:
        q_hat_E = 0.0

    # Corrected estimates on test set
    corrected_var = np.full(T, np.nan)
    corrected_es = np.full(T, np.nan)
    for t in range(n_cal, T):
        if not np.isnan(q_lo[t]):
            corrected_var[t] = -(q_lo[t] - q_hat_V)
        if not np.isnan(raw_es[t]):
            corrected_es[t] = raw_es[t] + q_hat_E

    return {'n_cal': n_cal, 'n_test': T-n_cal, 'q_hat_V': q_hat_V,
            'q_hat_E': q_hat_E, 'raw_var': raw_var, 'raw_es': raw_es,
            'corrected_var': corrected_var, 'corrected_es': corrected_es,
            'realized': realized}

print('Conformal function defined.')


Conformal function defined.


## 4. Results: Cross-Method Comparison

Pre-computed results from GARCH/EWMA pickle files and LLM simulations.


In [3]:
# Pre-computed results (from run_conformal_garch_ewma.py)
results = {
    'GARCH-N(250)':  {'raw_pi': 0.0031, 'q_V': +0.0040, 'corr_pi': 0.0080, 'green': '7/9', 'z2_pass': '9/9'},
    'GAS-N(250)':    {'raw_pi': 0.0031, 'q_V': +0.0043, 'corr_pi': 0.0088, 'green': '7/9', 'z2_pass': '9/9'},
    'GAS-t(250)':    {'raw_pi': 0.0012, 'q_V': -0.0066, 'corr_pi': 0.0211, 'green': '6/9', 'z2_pass': '9/9'},
    'GARCH-LPA':     {'raw_pi': 0.0061, 'q_V': +0.0106, 'corr_pi': 0.0029, 'green': '9/9', 'z2_pass': '9/9'},
    'EWMA-N(120)':   {'raw_pi': 0.0111, 'q_V': +0.0118, 'corr_pi': 0.0032, 'green': '9/9', 'z2_pass': '9/9'},
    'EWMA-DCS(120)': {'raw_pi': 0.0039, 'q_V': +0.0023, 'corr_pi': 0.0025, 'green': '9/9', 'z2_pass': '9/9'},
    'GPT-3.5+CP':    {'raw_pi': 0.0040, 'q_V': +0.0020, 'corr_pi': 0.0060, 'green': '8/9', 'z2_pass': '9/9'},
    'GPT-4+CP':      {'raw_pi': 0.0860, 'q_V': +0.0240, 'corr_pi': 0.0020, 'green': '9/9', 'z2_pass': '9/9'},
    'GPT-4o+CP':     {'raw_pi': 0.0570, 'q_V': +0.0200, 'corr_pi': 0.0020, 'green': '9/9', 'z2_pass': '9/9'},
}

print(f"{'Method':<18} {'Raw pi':>8} {'q_V':>10} {'Corr pi':>8} {'Green':>7} {'Z2 Pass':>8}")
print('-' * 65)
for name, r in results.items():
    print(f"{name:<18} {r['raw_pi']:>8.4f} {r['q_V']:>+10.4f} {r['corr_pi']:>8.4f} {r['green']:>7} {r['z2_pass']:>8}")


Method               Raw pi        q_V  Corr pi   Green  Z2 Pass
-----------------------------------------------------------------
GARCH-N(250)       0.0031    +0.0040   0.0080     7/9      9/9
GAS-N(250)         0.0031    +0.0043   0.0088     7/9      9/9
GAS-t(250)         0.0012    -0.0066   0.0211     6/9      9/9
GARCH-LPA          0.0061    +0.0106   0.0029     9/9      9/9
EWMA-N(120)        0.0111    +0.0118   0.0032     9/9      9/9
EWMA-DCS(120)      0.0039    +0.0023   0.0025     9/9      9/9
GPT-3.5+CP         0.0040    +0.0020   0.0060     8/9      9/9
GPT-4+CP           0.0860    +0.0240   0.0020     9/9      9/9
GPT-4o+CP          0.0570    +0.0200   0.0020     9/9      9/9


## 5. Key Figure: Conformal Correction Across Forecaster Classes


In [4]:
method_names = list(results.keys())
short_names = ['GARCH-N', 'GAS-N', 'GAS-t', 'GARCH-LPA', 'EWMA-N', 'EWMA-DCS',
               'GPT-3.5', 'GPT-4', 'GPT-4o']
mean_qvs = [r['q_V'] for r in results.values()]
colors = ['#2166AC']*4 + ['#4DAF4A']*2 + ['#E31A1C']*3

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(method_names))
bars = ax.bar(x, mean_qvs, color=colors, width=0.6, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, mean_qvs):
    y = bar.get_height()
    offset = 0.0005 if val >= 0 else -0.0012
    va = 'bottom' if val >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width()/2., y + offset,
            f'{val:+.4f}', ha='center', va=va, fontsize=8.5, fontweight='bold')

ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right', fontsize=10)
ax.set_ylabel(r'Mean $\hat{q}_V$ (conformal correction)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_elements = [
    Patch(facecolor='#2166AC', edgecolor='black', label='Parametric (GARCH/GAS)'),
    Patch(facecolor='#4DAF4A', edgecolor='black', label='Semi-parametric (EWMA)'),
    Patch(facecolor='#E31A1C', edgecolor='black', label='LLM-based'),
]
ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.22),
          ncol=3, fontsize=9.5, frameon=False)
fig.tight_layout()
fig.subplots_adjust(bottom=0.25)
plt.savefig('fig_conformal_agnostic.pdf', dpi=300, transparent=True, bbox_inches='tight')
plt.show()
print('Figure saved: fig_conformal_agnostic.pdf')


## 6. Interpretation

The conformal correction magnitude $\hat{q}_V$ serves as a **universal diagnostic** of tail miscalibration:

| Forecaster | Mean $\hat{q}_V$ | Interpretation |
|---|---|---|
| EWMA-DCS | +0.002 | Near-perfect calibration |
| GARCH-N / GAS-N | +0.004 | Small thin-tail correction (Normal assumption) |
| GAS-t | **-0.007** | Over-conservative (Student-t tails too heavy) |
| GARCH-LPA / EWMA-N | +0.011 | Moderate correction |
| GPT-4o | **+0.020** | Large — RLHF tail compression |
| GPT-4 | **+0.024** | Largest — severe RLHF tail compression |

LLM forecasters require corrections **5-6x larger** than parametric benchmarks,
confirming the framework's forecaster-agnosticism.
